//author: pdominguez                    
//date of creation/last modification: 2/1/2026               
//filesource: `tran_linearity_termoless.raw`                  
//tb_schem: `tb_linearity_8xPI_top.sch`               

# Presentación de resultados: linealidad del 8xPI sin termo ni 2to4

En esta notebook voy a presentar los resultados que se obtuvieron de la simulación del 8xPI que hizo Valentín, pero sin incluir los módulos del código termométrico y el 2to4. En lugar de eso, se generan los clocks de referencia con un 4to4 y el código termométrico se genera con estímulos "a mano".

Los resultados de simulación que se van a graficar en este archivo están en la salida de simulación `tran_linearity_termoless.raw`.                   

In [ ]:
### Seccion de imports

import os
import sys
from pathlib import Path

root_env = os.environ.get("CDR4176_ROOT")
if root_env:
    repo_root = Path(root_env).resolve()
else:
    repo_root = Path.cwd().resolve()
    while repo_root != repo_root.parent:
        if (repo_root / "project.yml").exists() or (repo_root / "CDR4176-main").is_dir():
            break
        repo_root = repo_root.parent
    if not ((repo_root / "project.yml").exists() or (repo_root / "CDR4176-main").is_dir()):
        raise RuntimeError("Could not locate repository root containing project.yml or CDR4176-main; set CDR4176_ROOT")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from scripts.notebook_data import resolve_raw_path

NOTEBOOK_ID = "simulations/tb_linearity_8xpi/results_tb_linearity_termoless.ipynb"

import matplotlib.pyplot as plt
import numpy as np

### Modificación de la función readRaw()

A continuación, se encuentra la nueva versión de la función `readRaw()`. El canbio con respecto a la versión anterior es que se agregó la capacidad de extraer la columna de valores de time (en el caso de transient analysis) o dc (en el caso de dc analysis).

In [ ]:
def readRaw(rawfile: str, variables: list[str]) -> dict[str,list[float]]:
    """
    readRaw()
    =======
    Lee un archivo raw y devuelve las variables indicadas en un
    diccionario. Las llaves son el nombre de las variables en
    la simulacion, y los valores son listas que contienen el resultado
    de simulacion de su respectiva variable.

    @author: pdominguez. Contactense ante cualquier duda.

    Parameters
    ----------

    rawfile: str
             Archivo donde se almacenan las salidas de simulacion de ngspice. 
             El archivo debe estar configurado en filetype = ascii.
                    
    variables: list[str]
               Lista que contiene las llaves del diccionario de salida.
               Las llaves deben ser iguales a los nombres de las senales en la salida ascii.

    Returns
    -------

    dict[str,list[float]]
            Diccionario donde las llaves son strings iguales al nombre de la senal en simulacion, 
            y el valor es la lista de floats que contiene los valores simulados respectivos a esa 
            variable

    Examples of use
    ---------------

    Se supone que el archivo out_simul.raw es una salida de simulacion de ngspice configurada en 
    filetype = ascii, y tiene las siguientes senales con sus respectivos valores:
    
    >>> Variables:
	>>> 0	time	time
	>>> 1	v(vout)	voltage
	>>> 2	v(vout8i)	voltage
	>>> 3	v(vini)	voltage
    >>> Values:
    >>>  0 	0.000000000000000e+00
    >>> 	1.472053331554910e-08
    >>> 	1.472053337682925e-08
    >>> 	0.000000000000000e+00

    >>>  1	1.000000000000000e-13
    >>> 	1.493710519767057e-08
    >>> 	1.493710531305648e-08
    >>> 	4.800000000000000e-03

    >>>  2	2.000000000000000e-13
    >>> 	1.533147593858479e-08
    >>> 	1.533147614939112e-08
    >>> 	9.600000000000001e-03

    ...y demas puntos

    Se podria hacer:

    >>> variables = ['time','v(vout)','v(vout8i)','v(vini)']   
    >>> resultados = readRaw('out_simul.raw',variables)

    y luego:

    >>> time = resultados['time']
    >>> vout = resultados['v(vout)']

    y siguiendo con las otras senales.
    """
    out_dict = {}
    aux_dict = {}
    head = None

    raw_path = resolve_raw_path(rawfile, NOTEBOOK_ID)
    with open(raw_path, 'r') as f:
        for line in f:
            linea = line.strip()
            #Flag para identificar la cabecera con las variables
            if(linea == 'Variables:'):
                head = True
            elif(linea == 'Values:'):
                head = False
            
            #Procesamiento de la cabecera y los datos
            if(head == True and linea != 'Variables:'):
                lin_split = linea.split()
                if lin_split[1] in variables:
                    aux_dict[int(lin_split[0])] = lin_split[1]
                    out_dict[lin_split[1]] = []

            elif(head == False and linea != 'Values:'):
                if(len(linea.split()) > 1):  #Nuevo instante de simulacion
                    offset = 0
                if offset in aux_dict:
                    if (offset != 0):
                        label = aux_dict[offset]
                        out_dict[label].append(float(linea))
                    else:
                        label = aux_dict[offset]
                        out_dict[label].append(float(linea.split()[1]))
                offset = offset + 1
    
    return out_dict

In [ ]:
# Lectura de archivo

variables = ['time','v(vout)','v(vout8i)','v(vini)','v(vouti1)','v(vouti2)','v(voutq1)','v(voutib1)','v(voutqb1)']
resultados = readRaw('tran_linearity_termoless.raw',variables)

In [ ]:
### Duración de cada cuadrante
LEN_Q = 20e-9
### Offset de captura de resultados
OFFS = 0
offs_time = np.abs(np.array(resultados['time'], dtype=float) - OFFS).argmin()

# Primer cuadrante
len_timeQ1 = np.abs(np.array(resultados['time'], dtype=float) - LEN_Q - OFFS).argmin()
vout_Q1 = resultados['v(vout)'][offs_time:len_timeQ1]
v8i_Q1 = resultados['v(vout8i)'][offs_time:len_timeQ1]
vini_Q1 = resultados['v(vini)'][offs_time:len_timeQ1]
vouti1_Q1 = resultados['v(vouti1)'][offs_time:len_timeQ1]
voutq1_Q1 = resultados['v(voutq1)'][offs_time:len_timeQ1]
voutib1_Q1 = resultados['v(voutib1)'][offs_time:len_timeQ1]
voutqb1_Q1 = resultados['v(voutqb1)'][offs_time:len_timeQ1]
vouti2_Q1 = resultados['v(vouti2)'][offs_time:len_timeQ1]
time_Q1 = resultados['time'][offs_time:len_timeQ1]

# Segundo cuadrante
len_timeQ2 = np.abs(np.array(resultados['time'], dtype=float) - 2*LEN_Q - OFFS).argmin()
vout_Q2 = resultados['v(vout)'][len_timeQ1:len_timeQ2]
v8i_Q2 = resultados['v(vout8i)'][len_timeQ1:len_timeQ2]
vini_Q2 = resultados['v(vini)'][len_timeQ1:len_timeQ2]
vouti1_Q2 = resultados['v(vouti1)'][len_timeQ1:len_timeQ2]
voutq1_Q2 = resultados['v(voutq1)'][len_timeQ1:len_timeQ2]
voutib1_Q2 = resultados['v(voutib1)'][len_timeQ1:len_timeQ2]
voutqb1_Q2 = resultados['v(voutqb1)'][len_timeQ1:len_timeQ2]
vouti2_Q2 = resultados['v(vouti2)'][len_timeQ1:len_timeQ2]
time_Q2 = resultados['time'][len_timeQ1:len_timeQ2]

# Tercer cuadrante
len_timeQ3 = np.abs(np.array(resultados['time'], dtype=float) - 3*LEN_Q - OFFS).argmin()
vout_Q3 = resultados['v(vout)'][len_timeQ2:len_timeQ3]
v8i_Q3 = resultados['v(vout8i)'][len_timeQ2:len_timeQ3]
vini_Q3 = resultados['v(vini)'][len_timeQ2:len_timeQ3]
vouti1_Q3 = resultados['v(vouti1)'][len_timeQ2:len_timeQ3]
voutq1_Q3 = resultados['v(voutq1)'][len_timeQ2:len_timeQ3]
voutib1_Q3 = resultados['v(voutib1)'][len_timeQ2:len_timeQ3]
voutqb1_Q3 = resultados['v(voutqb1)'][len_timeQ2:len_timeQ3]
vouti2_Q3 = resultados['v(vouti2)'][len_timeQ2:len_timeQ3]
time_Q3 = resultados['time'][len_timeQ2:len_timeQ3]

# Cuarto cuadrante
len_timeQ4 = np.abs(np.array(resultados['time'], dtype=float) - 4*LEN_Q - OFFS).argmin()
vout_Q4 = resultados['v(vout)'][len_timeQ3:len_timeQ4]
v8i_Q4 = resultados['v(vout8i)'][len_timeQ3:len_timeQ4]
vini_Q4 = resultados['v(vini)'][len_timeQ3:len_timeQ4]
vouti1_Q4 = resultados['v(vouti1)'][len_timeQ3:len_timeQ4]
voutq1_Q4 = resultados['v(voutq1)'][len_timeQ3:len_timeQ4]
voutib1_Q4 = resultados['v(voutib1)'][len_timeQ3:len_timeQ4]
voutqb1_Q4 = resultados['v(voutqb1)'][len_timeQ3:len_timeQ4]
vouti2_Q4 = resultados['v(vouti2)'][len_timeQ3:len_timeQ4]
time_Q4 = resultados['time'][len_timeQ3:len_timeQ4]

# Limites de los cuadrantes
print(time_Q1[0])
print(time_Q1[-1])
print(time_Q2[0])
print(time_Q2[-1])
print(time_Q3[0])
print(time_Q3[-1])
print(time_Q4[0])
print(time_Q4[-1])

### Función para graficar cada cuadrante

Cada cuadrante está dividido en 8 segmentos que corresponden a cada una de las 8 fases que "entran" dentro del cuadrante. A continuación, voy a escribir una función que, dandole el vector de datos en el tiempo de un cuadrante, grafique las 8 fases superpuestas en una misma ventana de tiempo.          

Para la simulación que yo planteé, cada cuadrante dura 20ns, por ende, cada una de las 8 fases del cuadrante dura 2.5ns. Pasado a ciclos de reloj, estos serían 5 ciclos de reloj para cada valor de fase.     

No obstante, para poder apreciar el desfase entre las señales correctamente, se debería graficar un solo flanco ascendente en la ventana de tiempo.

In [ ]:
def graph_QUADR(quadr_out: list[float],
                quadr_time: list[float],
                quadr: int,
                quadr_label: list[str],
                subplot: plt.Axes) -> None:
    """
    graph_QUADR()
    ============
    Funcion para graficar todas las fases de un solo cuadrante superpuestas, hablando en
    el contexto de un PI de 4 cuadrantes.

    Para los cuadrantes 1 y 3, la simulacion deberia recorrer las fases en sentido
    antihorario. Para los cuadrantes 2 y 4, en canbio, la simulacion debe hacer el sweep
    de fase en sentido horario. 
    
    Esto es asi por la misma naturaleza del hardware que se diseno y como se ingresan los
    estimulos. El codigo termometrico hace un sweep periodico con periodo de un cuadrante. 

    @author: pdominguez - Contactense sin problemas en caso de tener dudas. La funcion esta 
    implementada con algunas cantidades "hardcodeadas". Son libres de copiar esta funcion en
    su codigo y modificar estas cantidades para ajustarlas a su simulacion.

    Parameters
    ---------

    quadr_out: list[float]
               senal de salida de simulacion.

    quadr_time: list[float]
                senal de tiempo de simulacion,

    quadr: integer
           numero de cuadrante al que pertenece la senal,

    quadr_label: list[str]
                 limites de fase que delimitan el cuadrante. 

    subplot: plt.Axes
             subplot (objeto axes de matplotlib) donde se grafica. 
    """
    for i in np.arange(0,8):
        if (quadr == 1 or quadr ==3):
            len_time_seg1 = np.abs(quadr_time - (i+1)*2.5e-9 - (quadr-1)*20e-9).argmin()
            len_time_seg0 = np.abs(quadr_time - (i)*2.5e-9 - (quadr-1)*20e-9).argmin()
            subplot.plot(quadr_time[len_time_seg0:len_time_seg1]-i*2.5e-9,
                     quadr_out[len_time_seg0:len_time_seg1],
                     label = f'{i}{quadr_label[0]}+{8-i}{quadr_label[1]}')
        elif (quadr == 2 or quadr == 4):
            len_time_seg1 = np.abs(quadr_time - (7-i+1)*2.5e-9 - (quadr-1)*20e-9).argmin()
            len_time_seg0 = np.abs(quadr_time - (7-i)*2.5e-9 - (quadr-1)*20e-9).argmin()
            subplot.plot(quadr_time[len_time_seg0:len_time_seg1]-(7-i)*2.5e-9,
                     quadr_out[len_time_seg0:len_time_seg1],
                     label = f'{i}{quadr_label[0]}+{8-i}{quadr_label[1]}')
        #print(f'len_time_seg0:{len_time_seg0}')
        #print(f'len_time_seg1:{len_time_seg1}')
        #print(f'quadr_time[len_time_seg0]:{quadr_time[len_time_seg0]}')
        #print(f'quadr_time[len_time_seg1]:{quadr_time[len_time_seg1]}')
        
        subplot.grid(True)
        subplot.legend()

In [ ]:
# Generación de ventana de gráficos
results = plt.figure(figsize=[10,24])
quadrant1 = results.add_subplot(4,1,1)
quadrant2 = results.add_subplot(4,1,2)
quadrant3 = results.add_subplot(4,1,3)
quadrant4 = results.add_subplot(4,1,4)

# Primer cuadrante
graph_QUADR(vout_Q1,time_Q1,1,['Q','I'],quadrant1)
quadrant1.set_xlim(1e-9, 2e-9)
quadrant1.set_title('Sweep de fase en el primer cuadrante')

# Segundo cuadrante
graph_QUADR(vout_Q2,time_Q2,2,['IB','Q'],quadrant2)
quadrant2.set_xlim(1e-9+LEN_Q, 2e-9+LEN_Q)
quadrant2.set_title('Sweep de fase en el segundo cuadrante')

# Tercer cuadrante
graph_QUADR(vout_Q3,time_Q3,3,['QB','IB'],quadrant3)
quadrant3.set_xlim(1e-9+2*LEN_Q, 2e-9+2*LEN_Q)
quadrant3.set_title('Sweep de fase en el tercer cuadrante')

# Cuarto cuadrante
graph_QUADR(vout_Q4,time_Q4,4,['I','QB'],quadrant4)
quadrant4.set_xlim(1e-9+3*LEN_Q, 2e-9+3*LEN_Q)
quadrant4.set_title('Sweep de fase en el cuarto cuadrante')

results.tight_layout()

## Validación del método (o la función) con la que se grafican los resultados

Para tener una idea de si la función está bien implementada y no introduce desviaciones de fase en el acto de superponer las señales, se utilizó esta misma función para graficar un estímulo idealmente periódico en la simulación.                      
Este estímulo es el clock I que se genera con un estímulo PULSE de ngspice. Esta señal en la simulación es la que tiene el label de vinI.

In [ ]:
# Generación de ventana de gráficos
results8i = plt.figure(figsize=[10,24])
quadrant18i = results8i.add_subplot(4,1,1)
quadrant28i = results8i.add_subplot(4,1,2)
quadrant38i = results8i.add_subplot(4,1,3)
quadrant48i = results8i.add_subplot(4,1,4)

# Primer cuadrante
graph_QUADR(vini_Q1,time_Q1,1,['I','Q'],quadrant18i)
quadrant18i.set_xlim(1e-9, 2e-9)
quadrant18i.set_title('Calibración primer cuadrante señal de estímulo clock I')

# Segundo cuadrante
graph_QUADR(vini_Q2,time_Q2,2,['Q','IB'],quadrant28i)
quadrant28i.set_xlim(1e-9+LEN_Q, 2e-9+LEN_Q)
quadrant28i.set_title('Calibración segundo cuadrante señal de estímulo clock I')

# Tercer cuadrante
graph_QUADR(vini_Q3,time_Q3,3,['IB','QB'],quadrant38i)
quadrant38i.set_xlim(1e-9+2*LEN_Q, 2e-9+2*LEN_Q)
quadrant38i.set_title('Calibración tercer cuadrante señal de estímulo clock I')

# Cuarto cuadrante
graph_QUADR(vini_Q4,time_Q4,4,['QB','I'],quadrant48i)
quadrant48i.set_xlim(1e-9+3*LEN_Q, 2e-9+3*LEN_Q)
quadrant48i.set_title('Calibración cuarto cuadrante señal de estímulo clock I')

results8i.tight_layout()

### Ploteo de la fase 8I en la salida del 8xPI

En la simulación, se incluyó una instancia del 8xPI a la que se le variaron las entradas de control para obtener las variaciones de fase en la salida ya graficadas más arriba. No obstante, también se incluyó una instancia a la que no se le modificó las entradas de control, manteniendolas fijas para que todo el tiempo genere la fase 8I.                   

Se hizo la misma verificación que con la señal vinI graficada justo arriba, pero ahora con la salida del 8xPI que debería mantener la fase constante a lo largo de toda la simulación. El resultado es simular, más allá de alguna distorsión en los transitorios producto de las capacitancias del diseño, las 8 salidas se grafican en fase dejando una sola traza.


In [ ]:
# Generación de ventana de gráficos
results8i = plt.figure(figsize=[10,24])
quadrant18i = results8i.add_subplot(4,1,1)
quadrant28i = results8i.add_subplot(4,1,2)
quadrant38i = results8i.add_subplot(4,1,3)
quadrant48i = results8i.add_subplot(4,1,4)

# Primer cuadrante
graph_QUADR(v8i_Q1,time_Q1,1,['I','Q'],quadrant18i)
quadrant18i.set_xlim(1e-9, 2e-9)
quadrant18i.set_title('Calibración primer cuadrante fase 8I del 8xPI')

# Segundo cuadrante
graph_QUADR(v8i_Q2,time_Q2,2,['Q','IB'],quadrant28i)
quadrant28i.set_xlim(1e-9+LEN_Q, 2e-9+LEN_Q)
quadrant28i.set_title('Calibración segundo cuadrante fase 8I del 8xPI')

# Tercer cuadrante
graph_QUADR(v8i_Q3,time_Q3,3,['IB','QB'],quadrant38i)
quadrant38i.set_xlim(1e-9+2*LEN_Q, 2e-9+2*LEN_Q)
quadrant38i.set_title('Calibración tercer cuadrante fase 8I del 8xPI')

# Cuarto cuadrante
graph_QUADR(v8i_Q4,time_Q4,4,['QB','I'],quadrant48i)
quadrant48i.set_xlim(1e-9+3*LEN_Q, 2e-9+3*LEN_Q)
quadrant48i.set_title('Calibración cuarto cuadrante fase 8I del 8xPI')

results8i.tight_layout()

### Sensibilidad de las salidas del 4to4 a los cambios de fase en el 8xPI

La instancia del 8xPI que mantiene la fase constante, toma los clocks de referencia de un 4to4 y se ve que funciona perfecto (figura de arriba). Ahora bien, qué pasa con las salidas del 4to4 cuando está alimentando a un 8xPI que varía la fase? El 4to4 puede "ver" esos cambios desde la entrada del 8xPI? Qué tanto afectan las salidas del 4to4 esos cambios en la fase? Para contestar estas preguntas, se exportaron de la simulación las salidas del 4to4 que alimenta al 8xPI que cambia la fase.              

Voy a crear una nueva función basada en `graph_QUADR()` para que tenga más sentido utilizarla para graficar los clocks de referencia. La función `graph_QUADR()` está diseñada para graficar un solo clock y ver cómo va variando la fase a lo largo de un cuadrante. Yo no quiero ver eso ahora con los clocks, quiero graficarlos "uno encima de otro" y corroborar si dejan una sola traza o no.

In [ ]:
def clks_QUADR(quadr_out: list[float],
                quadr_time: list[float],
                quadr: int,
                clk_label: str,
                color: str,
                subplot: plt.Axes) -> None:
    """
    graph_QUADR()
    ============
    Funcion para graficar todas las fases de un solo cuadrante superpuestas, hablando en
    el contexto de un PI de 4 cuadrantes.

    Para los cuadrantes 1 y 3, la simulacion deberia recorrer las fases en sentido
    antihorario. Para los cuadrantes 2 y 4, en canbio, la simulacion debe hacer el sweep
    de fase en sentido horario. 
    
    Esto es asi por la misma naturaleza del hardware que se diseno y como se ingresan los
    estimulos. El codigo termometrico hace un sweep periodico con periodo de un cuadrante. 

    @author: pdominguez - Contactense sin problemas en caso de tener dudas. La funcion esta 
    implementada con algunas cantidades "hardcodeadas". Son libres de copiar esta funcion en
    su codigo y modificar estas cantidades para ajustarlas a su simulacion.

    Parameters
    ---------

    quadr_out: list[float]
               senal de salida de simulacion.

    quadr_time: list[float]
                senal de tiempo de simulacion,

    quadr: integer
           numero de cuadrante al que pertenece la senal,

    quadr_label: list[str]
                 limites de fase que delimitan el cuadrante. 

    subplot: plt.Axes
             subplot (objeto axes de matplotlib) donde se grafica. 
    """
    for i in np.arange(0,8):
        if (quadr == 1 or quadr ==3):
            len_time_seg1 = np.abs(quadr_time - (i+1)*2.5e-9 - (quadr-1)*20e-9).argmin()
            len_time_seg0 = np.abs(quadr_time - (i)*2.5e-9 - (quadr-1)*20e-9).argmin()
            if (i == 0):
                subplot.plot(quadr_time[len_time_seg0:len_time_seg1]-i*2.5e-9,
                     quadr_out[len_time_seg0:len_time_seg1],
                     f'{color}',
                     label = f'Fase {clk_label}')
            else:
                subplot.plot(quadr_time[len_time_seg0:len_time_seg1]-i*2.5e-9,
                     quadr_out[len_time_seg0:len_time_seg1],
                     f'{color}')

        elif (quadr == 2 or quadr == 4):
            len_time_seg1 = np.abs(quadr_time - (7-i+1)*2.5e-9 - (quadr-1)*20e-9).argmin()
            len_time_seg0 = np.abs(quadr_time - (7-i)*2.5e-9 - (quadr-1)*20e-9).argmin()
            if (i == 0):
                subplot.plot(quadr_time[len_time_seg0:len_time_seg1]-(7-i)*2.5e-9,
                     quadr_out[len_time_seg0:len_time_seg1],
                     f'{color}',
                     label = f'Fase {clk_label}')
            else:
                subplot.plot(quadr_time[len_time_seg0:len_time_seg1]-(7-i)*2.5e-9,
                     quadr_out[len_time_seg0:len_time_seg1],
                     f'{color}')

        #print(f'len_time_seg0:{len_time_seg0}')
        #print(f'len_time_seg1:{len_time_seg1}')
        #print(f'quadr_time[len_time_seg0]:{quadr_time[len_time_seg0]}')
        #print(f'quadr_time[len_time_seg1]:{quadr_time[len_time_seg1]}')
        
        subplot.grid(True)
        subplot.legend()

In [ ]:
# Generación de ventana de gráficos
results8i = plt.figure(figsize=[10,24])
quadrant18i = results8i.add_subplot(4,1,1)
quadrant28i = results8i.add_subplot(4,1,2)
quadrant38i = results8i.add_subplot(4,1,3)
quadrant48i = results8i.add_subplot(4,1,4)

# Primer cuadrante
clks_QUADR(vouti1_Q1,time_Q1,1,'I','r',quadrant18i)
clks_QUADR(voutq1_Q1,time_Q1,1,'Q','g',quadrant18i)
clks_QUADR(voutib1_Q1,time_Q1,1,'IB','b',quadrant18i)
clks_QUADR(voutqb1_Q1,time_Q1,1,'QB','k',quadrant18i)
quadrant18i.set_xlim(1e-9, 2e-9)
quadrant18i.set_title('Salidas del 4to4 para el primer cuadrante')

# Segundo cuadrante
clks_QUADR(vouti1_Q2,time_Q2,2,'I','r',quadrant28i)
clks_QUADR(voutq1_Q2,time_Q2,2,'Q','g',quadrant28i)
clks_QUADR(voutib1_Q2,time_Q2,2,'IB','b',quadrant28i)
clks_QUADR(voutqb1_Q2,time_Q2,2,'QB','k',quadrant28i)
quadrant28i.set_xlim(1e-9+LEN_Q, 2e-9+LEN_Q)
quadrant28i.set_title('Salidas del 4to4 para el segundo cuadrante')

# Tercer cuadrante
clks_QUADR(vouti1_Q3,time_Q3,3,'I','r',quadrant38i)
clks_QUADR(voutq1_Q3,time_Q3,3,'Q','g',quadrant38i)
clks_QUADR(voutib1_Q3,time_Q3,3,'IB','b',quadrant38i)
clks_QUADR(voutqb1_Q3,time_Q3,3,'QB','k',quadrant38i)
quadrant38i.set_xlim(1e-9+2*LEN_Q, 2e-9+2*LEN_Q)
quadrant38i.set_title('Salidas del 4to4 para el tercer cuadrante')

# Cuarto cuadrante
clks_QUADR(vouti1_Q4,time_Q4,4,'I','r',quadrant48i)
clks_QUADR(voutq1_Q4,time_Q4,4,'Q','g',quadrant48i)
clks_QUADR(voutib1_Q4,time_Q4,4,'IB','b',quadrant48i)
clks_QUADR(voutqb1_Q4,time_Q4,4,'QB','k',quadrant48i)
quadrant48i.set_xlim(1e-9+3*LEN_Q, 2e-9+3*LEN_Q)
quadrant48i.set_title('Salidas del 4to4 para el cuarto cuadrante')

results8i.tight_layout()

Es evidente que el 4to4 puede "ver" los cambios de fase del 8xPI porque las salidas activas en cada cuadrante (un par por cada cuadrante) muestran una deriva de fase a lo largo del tiempo, o bien, a medida que va modificandosé la fase del 8xPI.

## Medición del desfase de cada salida

El objetivo de hacer esta simulación, además de graficar de una manera específica los resultados, es obtener mediciones precisas del desfase que hay en las salidas del clock. Para poder calcular esto, se va a utilizar una nueva función que registre el tiempo en el que ocurren los flancos ascendentes dentro de un cuadrante de la simulación.       

Esta función va a tener como entrada la salida de simulación y el tiempo en ese cuadrante, y va a tener como salida una lista de valores que corresponden al desfase medido.   

El desfase se va a medir con respecto a la salida del 8xPI generada con la instancia que mantiene la fase 8I en todo momento.

In [ ]:
def phase_meas(quadr_out: list[float],
               quadr_8i: list[float],
               quadr_time: list[float],
               threshold: float,
               Ts: float,
               quadr: int
               ) -> list[float]:

    ph_out = []

    for i in np.arange(0,8):
        if (quadr == 1 or quadr ==3):
            start_samp = np.abs(quadr_time - (i)*2.5e-9 - (quadr-1)*20e-9 - 1e-9).argmin()
            time8I_samp = start_samp
            while(quadr_8i[time8I_samp] < threshold):
                time8I_samp = time8I_samp+1
            time8I = quadr_time[time8I_samp]
            
            time_ph_samp = time8I_samp
            if (quadr_out[time_ph_samp] >= threshold):
                while (quadr_out[time_ph_samp] >= threshold):
                    time_ph_samp = time_ph_samp+1
            
            while (quadr_out[time_ph_samp] < threshold):
                time_ph_samp = time_ph_samp+1

            time_ph = quadr_time[time_ph_samp]
            diff_ph = round((time_ph - time8I)*360/Ts, ndigits=5)
            if diff_ph == 360: diff_ph = 0
            
            ph_out.append(diff_ph)
            
        elif (quadr == 2 or quadr == 4):
            start_samp = np.abs(quadr_time - (7-i)*2.5e-9 - (quadr-1)*20e-9 - 1e-9).argmin()
            time8I_samp = start_samp
            while(quadr_8i[time8I_samp] < threshold):
                time8I_samp = time8I_samp+1
            time8I = quadr_time[time8I_samp]
            
            time_ph_samp = time8I_samp
            if (quadr_out[time_ph_samp] >= threshold):
                while (quadr_out[time_ph_samp] >= threshold):
                    time_ph_samp = time_ph_samp+1
            
            while (quadr_out[time_ph_samp] < threshold):
                time_ph_samp = time_ph_samp+1

            time_ph = quadr_time[time_ph_samp]
            diff_ph = round((time_ph - time8I)*360/Ts, ndigits=5)
            if diff_ph == 360: diff_ph = 0

            ph_out.append(diff_ph)
    
    return ph_out
            

In [ ]:
phase_Q1 = phase_meas(vout_Q1,v8i_Q1,time_Q1,0.6,0.5e-9,1)
print(phase_Q1)
phase_Q2 = phase_meas(vout_Q2,v8i_Q2,time_Q2,0.6,0.5e-9,2)
print(phase_Q2)
phase_Q3 = phase_meas(vout_Q3,v8i_Q3,time_Q3,0.6,0.5e-9,3)
print(phase_Q3)
phase_Q4 = phase_meas(vout_Q4,v8i_Q4,time_Q4,0.6,0.5e-9,4)
print(phase_Q4)

In [ ]:
phase_sweep = list(dict.fromkeys(phase_Q1+phase_Q2+phase_Q3+phase_Q4))
print(len(phase_sweep))

EN ESTA SECCIÓN DESCRIBO UN PROBLEMA YA SOLUCIONADO

Una conclusión importante que se obtiene de calcular la longitud de fases no repetidas que puede generar en la salida el PI es que hay solo 30 valores y no 32. Porqué se produce esto? Primero que nada, porque el bit más significativo del código termométrico está conectado a VSS en mi simulación. Esto ocasiona que la fase 8Q y 8QB no sean accesibles por el PI, y cuando se hace el sweep de fase, las cantidades más cercanas son 7IB+Q y 7IB+QB, respectivamente.          

Esto es un problema, porque hace que el PI sea de 32 pasos realmente y tenga un "salto" de fase en esos cambios de cuadrante.           

Una de las alternativas que se me había ocurrido es conectar el bit menos significativo del selector de cuadrante a la entrada más significariva del código termo. Esto mejora el desempeño parcialmente porque se pueden alcanzar 31 fases, pero no alcanza las 32 fases que podría lograr el PI con un control adecuado.     

Como referencia, el selector de cuadrante se comporta de la siguiente manera:

* Cuadrante 1: 2'b00
* Cuadrante 2: 2'b01
* Cuadrante 3: 2'b11
* Cuadrante 4: 2'b10

Explico esto porque, la solución que encontré para recorrer las 4 fases es que el bit más significativo del código termométrico vaya alternando de 0 a 1 en el paso de cuadrante. Esto sería, 0 en los cuadrantes 1 y 3, y 1 en los cuadrantes 2 y 4. Del selector de cuadrante, ese patrón se puede lograr con una compuerta XOR (3 NAND) y 2 inversores (los inversores para generar las entradas negadas).      

Hay una implementación alternativa de compuertas XOR que utiliza 4 compuertas NAND y no necesita inversores. Al fin y al cabo, la cantidad de transistores va a ser la misma (8 PMOS y 8 NMOS). 


### Medición (o cálculo) de la INL y la DNL

La DNL y la INL son especificaciones que hablan de la linealidad de conversión de una entrada digital de bits (entradas de control del PI) a una salida analógica de fase (salida del PI). Para calcular cada una voy a hacer una función en la que ingrese el sweep de fase y egrese una lista con los valores de la INL y DNL para cada entrada de control.              

Para poner un contexto, la INL (Integral NonLinearity) es la diferencia que hay entre cada valor de fase con respecto al valor ideal lineal que se busca que haya en un conversor. Esta se puede expresar en grados (porque sería una diferencia de fases al fin y al cabo) o en LSB (Less Significant Bit), normalizando la cantidad anterior en grados al mínimo cambio que se puede efectuar en las entradas del conversor.

Por otro lado, la DNL (Differencial NonLinearity) se define como la diferencia que existe entre el salto de fase de la salida al siguiente valor con respecto al salto ideal.          

Es claro entonces, que para calcular ambas, primero hay que definir cuál sería la característica de transferencia ideal. Esto es sencillo, si el rango en la salida es de 360° y hay 32 valores posibles de fase, cada valor tendría que ser 11.25° mayor que el anterior para hacer una excursión completamente lineal. En la siguiente celda se genera y grafica la característica de transferencia ideal con respecto a la real.

In [ ]:
phase_sweep_ideal = np.linspace(0,360,32, endpoint=False)
codigo = np.arange(0,32)

fig_ideal_sweep, ax_ideal_sweep = plt.subplots(figsize=[10,6])
#ax_ideal_sweep.plot(codigo,phase_sweep_ideal,'g')
ax_ideal_sweep.plot(codigo,phase_sweep_ideal,'gs',label='Característica ideal')
ax_ideal_sweep.plot(codigo,phase_sweep,'r^', label='Característica real')
ax_ideal_sweep.set_xlabel('Código')
ax_ideal_sweep.set_ylabel('Fase[°]')
ax_ideal_sweep.set_title('Característica ideal vs. Característica real')
ax_ideal_sweep.set_xticks([0,4,8,12,16,20,24,28,31])
ax_ideal_sweep.set_yticks([0,45,90,135,180,225,270,315,348.75])
ax_ideal_sweep.grid()
ax_ideal_sweep.legend()


In [ ]:
def calc_INL(ideal: list[float], real: list[float]) -> list[float]:
    
    INL = [0]*len(ideal)
    step = 360/len(ideal)
    for i in range(0,len(ideal)):
        INL[i] = (real[i] - ideal[i])/step
    
    return INL

In [ ]:
def calc_DNL(real: list[float]) -> list[float]:
    
    DNL = [0]*(len(real)-1)
    step = 360/len(real)
    for i in range(0,len(real)-1):
        DNL[i] = (real[i+1] - real[i] - step)/step
    
    return DNL

In [ ]:
INL = calc_INL(phase_sweep_ideal,phase_sweep)
print(INL)
DNL = calc_DNL(phase_sweep)
print(DNL)

In [ ]:
from scipy.interpolate import PchipInterpolator

INL_inter = PchipInterpolator(codigo,INL)
DNL_inter = PchipInterpolator(codigo[:-1],DNL)
fine_codigo_INL = np.linspace(codigo.min(),codigo.max(),10*len(codigo))
fine_codigo_DNL = np.linspace(codigo[0],codigo.max()-1,10*len(codigo))

fig_INL_DNL, ax_DNL_INL = plt.subplots(figsize=[10,6])
#ax_DNL_INL.plot(codigo,INL,'g')
ax_DNL_INL.plot(fine_codigo_INL,INL_inter(fine_codigo_INL),'g')
ax_DNL_INL.plot(codigo,INL,'go',label='INL')
ax_DNL_INL.plot(fine_codigo_DNL,DNL_inter(fine_codigo_DNL),'r', label='DNL')
ax_DNL_INL.plot(codigo[:-1],DNL,'ro')
ax_DNL_INL.set_xlabel('Código')
ax_DNL_INL.set_ylabel('LSB')
ax_DNL_INL.set_title('INL y DNL del 8xPI')
ax_DNL_INL.set_xticks([0,4,8,12,16,20,24,28,31])
#ax_DNL_INL.set_yticks([0,45,90,135,180,225,270,315,348.75])
ax_DNL_INL.grid()
ax_DNL_INL.legend()


### Tablas de interés

Voy a generar una tabla que diga plotee la información numérica de un solo vistazo

In [ ]:
columnas = ['Fase ideal[°]', 'Fase medida[°]', 'INL[LSB]','DNL[LSB]']

filas = []
for i in range(0,4):
    match i:
        case 0:
            for n in range(0,8):
                filas.append(f'{8-n}I+{n}Q')
        case 1:
            for n in range(0,8):
                filas.append(f'{8-n}Q+{n}IB')
        case 2:
            for n in range(0,8):
                filas.append(f'{8-n}IB+{n}QB')
        case 3:
            for n in range(0,8):
                filas.append(f'{8-n}QB+{n}I')

data = list(zip(phase_sweep_ideal,phase_sweep,np.round(INL,decimals=4),list(np.round(DNL,decimals=4))+['-']))

fig_table , ax_table = plt.subplots(figsize=[10,6])
ax_table.axis('off')

ax_table.table(data,colLabels=columnas,rowLabels=filas, loc='center')
